# PSN-2 Sequential Training — Kaggle

**Automatic D1 → D2 → D3 → D4 → D5 → D6 training with gate certification.**

This notebook trains all stages sequentially in a single run. Each stage:
1. Trains for the specified number of steps
2. Evaluates and checks gate certification
3. Automatically continues to next stage (or stops if gates fail)

**One-time setup (first session only):**
1. Upload `psn2_kaggle_full_v2.zip` as a Kaggle Dataset → name it `psn2-kaggle`
2. Go to **Account → Settings → API → Create New Token** → download `kaggle.json`
3. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` as **Notebook Secrets** (the 🔑 panel)
4. Attach the `psn2-kaggle` dataset to this notebook
5. Set GPU to **2×T4** or **P100**
6. Run all cells

**Session plan:**
- Session 1-2: D1 (20k steps, ~3-4 hours)
- Session 3-4: D2 (20k steps, ~3-4 hours)
- Session 5: D3 (15k steps, ~2-3 hours)
- Session 6-7: D4 (25k steps, ~4-5 hours)
- Session 8-9: D5 (30k steps, ~5-6 hours)
- Session 10+: D6 (40k steps, ~6-8 hours)

**Or run continuously:** Set `START_STAGE='D1'` and `END_STAGE='D6'` and let it run through all stages (requires Kaggle Notebooks+ for extended runtime).

In [ ]:
# ── 1. Secrets & Kaggle API credentials ─────────────────────────────────────
import os, sys, json, shutil, subprocess, atexit, signal, time
from pathlib import Path

# Load Kaggle credentials from Notebook Secrets
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    KAGGLE_USERNAME = _secrets.get_secret('KAGGLE_USERNAME')
    KAGGLE_KEY      = _secrets.get_secret('KAGGLE_KEY')
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY']      = KAGGLE_KEY
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / 'kaggle.json').write_text(
        json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY})
    )
    (kaggle_dir / 'kaggle.json').chmod(0o600)
    print(f'Kaggle credentials loaded for user: {KAGGLE_USERNAME}')
    KAGGLE_API_AVAILABLE = True
except Exception as e:
    print(f'WARNING: Kaggle secrets not found ({e})')
    print('Checkpoint auto-push disabled.')
    KAGGLE_USERNAME = None
    KAGGLE_API_AVAILABLE = False

CHECKPOINT_DATASET = 'psn2-checkpoint'

In [ ]:
# ── 2. Locate repo root ──────────────────────────────────────────────────────
WORKDIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train_sequential.py' in files or 'train.py' in files:
        WORKDIR = root
        break

assert WORKDIR, 'Could not find train.py. Attach psn2-kaggle dataset.'
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
os.chdir(WORKDIR)
print('Repo root:', WORKDIR)

In [ ]:
# ── 3. Install deps & verify GPU ─────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm', 'kaggle'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

In [ ]:
# ── 4. Checkpoint push/pull helpers ─────────────────────────────────────────
ARTIFACTS_DIR = '/kaggle/working/artifacts'
LATEST_CKPT   = f'{ARTIFACTS_DIR}/latest.pt'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

def push_checkpoint(reason='session_end'):
    """Push latest.pt to the psn2-checkpoint Kaggle dataset."""
    if not KAGGLE_API_AVAILABLE:
        print('  [push] Skipped — no Kaggle credentials')
        return
    if not os.path.exists(LATEST_CKPT):
        print('  [push] No checkpoint to push')
        return

    size_mb = os.path.getsize(LATEST_CKPT) / 1e6
    print(f'  [push] Pushing checkpoint ({size_mb:.1f} MB) — reason: {reason}')

    push_dir = '/kaggle/working/ckpt_push'
    os.makedirs(push_dir, exist_ok=True)
    shutil.copy(LATEST_CKPT, f'{push_dir}/latest.pt')

    meta = {
        'title': 'PSN-2 Checkpoint',
        'id': f'{KAGGLE_USERNAME}/{CHECKPOINT_DATASET}',
        'licenses': [{'name': 'other'}],
    }
    with open(f'{push_dir}/dataset-metadata.json', 'w') as f:
        json.dump(meta, f)

    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', push_dir,
         '-m', f'auto-push: {reason} at {time.strftime("%Y-%m-%d %H:%M")}',
         '--dir-mode', 'zip'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('  [push] Done:', result.stdout.strip())
    else:
        result2 = subprocess.run(
            ['kaggle', 'datasets', 'create', '-p', push_dir, '--dir-mode', 'zip'],
            capture_output=True, text=True
        )
        if result2.returncode == 0:
            print('  [push] Created new dataset:', result2.stdout.strip())
        else:
            print('  [push] FAILED:', result.stderr.strip(), result2.stderr.strip())

def pull_checkpoint():
    """Copy latest.pt from /kaggle/input/psn2-checkpoint/ if present."""
    for dirpath, _, files in os.walk('/kaggle/input'):
        if 'latest.pt' in files and 'psn2-kaggle' not in dirpath:
            src = os.path.join(dirpath, 'latest.pt')
            shutil.copy(src, LATEST_CKPT)
            size_mb = os.path.getsize(LATEST_CKPT) / 1e6
            print(f'  [pull] Loaded checkpoint from {src} ({size_mb:.1f} MB)')
            return True
    print('  [pull] No previous checkpoint found — starting fresh')
    return False

_push_registered = False
def _on_exit():
    push_checkpoint(reason='exit_handler')

def _on_signal(signum, frame):
    push_checkpoint(reason=f'signal_{signum}')
    sys.exit(1)

if not _push_registered:
    atexit.register(_on_exit)
    for sig in (signal.SIGTERM, signal.SIGINT):
        try:
            signal.signal(sig, _on_signal)
        except (OSError, ValueError):
            pass
    _push_registered = True
    print('Crash/exit handler registered')

In [ ]:
# ── 5. Pull checkpoint from previous session ─────────────────────────────────
if not os.path.exists(LATEST_CKPT):
    pull_checkpoint()
else:
    size_mb = os.path.getsize(LATEST_CKPT) / 1e6
    print(f'  Checkpoint already in working dir ({size_mb:.1f} MB)')

In [ ]:
# ── 6. Session config ────────────────────────────────────────────────────────
# EDIT THIS: set stage range for this session
START_STAGE = 'D1'   # D1 | D2 | D3 | D4 | D5 | D6
END_STAGE   = 'D6'   # D1 | D2 | D3 | D4 | D5 | D6

# Set to True to continue even if gates fail
SKIP_GATE_CHECK = False

print(f'Training stages: {START_STAGE} → {END_STAGE}')
print(f'Skip gate checks: {SKIP_GATE_CHECK}')

In [ ]:
# ── 7. Sequential training ───────────────────────────────────────────────────
resume_flag = ['--resume', LATEST_CKPT] if os.path.exists(LATEST_CKPT) else []
skip_gate_flag = ['--skip-gate-check'] if SKIP_GATE_CHECK else []

cmd = [
    sys.executable, 'train_sequential.py',
    '--config', 'configs/default.json',
    '--checkpoint-dir', ARTIFACTS_DIR,
    '--start-stage', START_STAGE,
    '--end-stage', END_STAGE,
] + resume_flag + skip_gate_flag

print('Running:', ' '.join(cmd))
print()

try:
    result = subprocess.run(cmd, cwd=WORKDIR)
    print('\nExit code:', result.returncode)
    if result.returncode != 0:
        print('Training exited with error — pushing checkpoint anyway')
        push_checkpoint(reason='training_error')
except KeyboardInterrupt:
    print('\nInterrupted — pushing checkpoint')
    push_checkpoint(reason='keyboard_interrupt')
    raise

In [ ]:
# ── 8. Push checkpoint ───────────────────────────────────────────────────────
push_checkpoint(reason='training_complete')

In [ ]:
# ── 9. Results summary ───────────────────────────────────────────────────────
results_file = os.path.join(ARTIFACTS_DIR, 'sequential_results.json')
if os.path.exists(results_file):
    with open(results_file) as f:
        results = json.load(f)
    
    print('\n' + '='*70)
    print('SEQUENTIAL TRAINING RESULTS')
    print('='*70 + '\n')
    
    for stage, data in results.items():
        status = '✓ PASSED' if data.get('passed', False) else '✗ FAILED'
        print(f'{stage}: {status}')
        
        eval_res = data.get('eval_results', {})
        if 'grid_accuracy' in eval_res:
            print(f'  Grid accuracy:       {eval_res["grid_accuracy"]:.4f}')
        if 'relation_prediction' in eval_res:
            print(f'  Relation prediction: {eval_res["relation_prediction"]:.4f}')
        print()
    
    print(json.dumps(results, indent=2))
else:
    print('No results file found')

In [ ]:
# ── 10. Artifacts summary ────────────────────────────────────────────────────
if os.path.exists(ARTIFACTS_DIR):
    print('\nArtifacts:')
    for f in sorted(os.listdir(ARTIFACTS_DIR)):
        path = os.path.join(ARTIFACTS_DIR, f)
        if os.path.isfile(path):
            size_mb = os.path.getsize(path) / 1e6
            print(f'  {f:45s}  {size_mb:.1f} MB')
else:
    print('No artifacts')